In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from src.data_add_column import  add_era_to_all, add_round_group, round_group_summary

In [2]:
# loading data

# import data
raw_data = pd.read_csv('../data/uefa_results.csv')
country_stats = pd.read_pickle('../data/processed/country_stats_complete.pkl')
country_season_highest = pd.read_pickle('../data/processed/country_season_highest.pkl')
country_season_competition_highest = pd.read_pickle('../data/processed/country_season_competition_highest.pkl')

In [3]:
# Era labels on all four DataFrames
rd, cs, csh, csch = add_era_to_all(
    raw_data, 
    country_stats,
    country_season_highest, 
    country_season_competition_highest,
    golden_start= "1987/88",
    golden_end = "1999/00",
)

# Round grouping on raw_data only
rd = add_round_group(rd)
round_group_summary(rd)   # quick sanity check


  Era definition : 1987/88  →  1999/00
  Era                     Seasons      First       Last
  ----------------------------------------------------
  Pre-Golden Era               18    1969/70    1986/87
  Golden Era                   13    1987/88    1999/00
  Post-Golden Era              26    2000/01    2025/26


  [round_group] All round labels mapped successfully (13872 / 13872 rows).

  Round group             Count       %
  --------------------------------------
  stage                  12,747   91.9%
  Round of 16               119    0.9%
  Quarter Finals            572    4.1%
  Semi Finals               288    2.1%
  Final                     146    1.1%
  TOTAL                  13,872  100.0%



# Step 2:  Q2: Golden Era country comparison

All metrics are normalised by `num_teams` to avoid quota-size bias. A four-pillar composite dominance score is computed for every country: tournament round depth (35 %), win rate (25 %), PPG per team (25 %), goal difference per game (15 %). Each pillar is min-max normalised before weighting. Italy's rank out of all available countries is reported. A Kruskal-Wallis test checks whether the top-5 countries are genuinely separated.

## Formal framework

We are now testing the following hypothesis:

- $H_{3}$ — During the Golden Era, Italy performed better than all other European countries across both match-level and tournament-depth metrics.

This step has two layers: the first is *descriptive*: rank every country on each metric and build a composite score. The second is *inferential*: test whether Italy's season-by-season distribution is statistically different from the rest of the field.     

## Metrics

| Pillar | Metric | Source | Normalisation |
|:-------|:-------|:--------------|:----|
|Match performance  | Win rate | `country_stats` -> `win_rate` | Already a rate |
|Match performance  | points per game (*PPG*) | `country_stats` -> `ppg_3`/`num_teams` | Quota size |
|Match performance  | Goal Difference per Game (*GDpG*) | `country_stats` -> `gdpg`/`num_teams`| Quota size |
|Tournament depth  | Avg deepest round | `country_stats` -> `highest_round` | -- |
|Tournament depth  | Finals + semis count | same | Scaled by season present |

## Composite score

Four pillars, min-max normalised within the Golden Era, then weighted:

```python
score = 0.25 · win_rate_norm
      + 0.25 · ppg3_pt_norm
      + 0.15 · gdpg_pt_norm
      + 0.35 · avg_round_norm
```

Tournament depth carries the most weight because reaching a Final is the clearest signal of dominance and cannot be inflated by entry quota.

## Statistical approach

- Mann-Whitney $U$ (one-tailed): Italy's season-by-season win rate vs the pooled distribution of all other countries 
- Rank consistency: Italy's average season rank across the Golden Era (lower = more consistently near the top)
- Both tests run per-metric and for the composite score

## Step-by-step 
1. Filter all four DataFrames to Golden Era only
2. Per-country match-level aggregates (`country_stats`)
3. Per-country tournament depth (`country_season_highest`)
4. Per-competition breakdown (`country_season_competition_highest`)
5. Composite score + final ranking
6. Statistical tests (Italy vs Europe)

### 1. Filter all four dataframe to Golden Era only

In [4]:
"""
Step 2a · Filter to Golden Era
───────────────────────────────
Slices all four DataFrames to the Golden Era only.
Every subsequent Step 2 function works on these slices.

Returns named results so it is obvious which DataFrame is which
throughout the rest of the pipeline.
"""

import pandas as pd

def filter_golden_era(
    cs:   pd.DataFrame,
    csh:  pd.DataFrame,
    csch: pd.DataFrame,
    rd:   pd.DataFrame | None = None,
    era_col: str = "era",
) -> dict[str, pd.DataFrame]:
    """
    Filter country_stats, country_season_highest,
    country_season_competition_highest (and optionally raw_data)
    to the Golden Era rows.

    Parameters
    ----------
    cs   : country_stats              (must have 'era' column)
    csh  : country_season_highest     (must have 'era' column)
    csch : country_season_competition_highest (must have 'era' column)
    rd   : raw_data                   (optional; pass None to skip)
    era_col : name of the era column  (default 'era')

    Returns
    -------
    dict with keys 'cs', 'csh', 'csch', and 'rd' (None if not passed)
    Each value is a reset-index copy filtered to Golden Era only.
    """
    
    # Era labels (single source of truth)
    ERA_PRE    = "Pre-Golden Era"
    ERA_GOLDEN = "Golden Era"
    ERA_POST   = "Post-Golden Era"

    
    inputs = {"cs": cs, "csh": csh, "csch": csch}
    if rd is not None:
        inputs["rd"] = rd

    result = {}
    print(f"\n  Filtering to: {ERA_GOLDEN}")
    print(f"  {'DataFrame':<40}  {'Rows in':>8}  {'Rows out':>9}")
    print("  " + "-" * 62)

    for key, df in inputs.items():
        if era_col not in df.columns:
            raise KeyError(
                f"'{era_col}' column missing from '{key}'. "
                "Run add_era_to_all() (step0_era.py) first."
            )
        filtered = (df[df[era_col] == ERA_GOLDEN]
                    .copy()
                    .reset_index(drop=True))
        result[key] = filtered
        print(f"  {key:<40}  {len(df):>8,}  {len(filtered):>9,}")

    if rd is None:
        result["rd"] = None

    # ── Coverage: seasons and countries present in Golden Era ────────────────
    ge_cs = result["cs"]
    seasons   = sorted(ge_cs["season"].unique(),
                        key=lambda s: int(s.split("/")[0]))
    countries = sorted(ge_cs["country"].unique())

    print(f"\n  Seasons  : {len(seasons)}  ({seasons[0]} → {seasons[-1]})")
    print(f"  Countries: {len(countries)}")

    return result

In [5]:
ge = filter_golden_era(cs, csh, csch,rd)


  Filtering to: Golden Era
  DataFrame                                  Rows in   Rows out
  --------------------------------------------------------------
  cs                                           3,363        767
  csh                                          3,363        767
  csch                                        16,815      3,835
  rd                                          13,872      2,191

  Seasons  : 13  (1987/88 → 1999/00)
  Countries: 59


In [6]:
ge['rd']

,season,competition,round,team_a,country_a,team_b,country_b,score_leg1,score_leg2,score_aggregate,extra_time,penalties,era,round_group
0,1987/88,CL,Round 1,Rapid Wien,Aut,Hamrun Spartans,Mlt,6-0,1-0,7-0,no,no,Golden Era,stage
1,1987/88,CL,Round 1,AGF Aarhus,Den,Jeunesse d'Esch,Lux,4-1,0-1,4-2,no,no,Golden Era,stage
2,1987/88,CL,Round 1,Shamrock Rovers,Irl,Omonia Nicosia,Cyp,0-1,0-0,0-1,no,no,Golden Era,stage
3,1987/88,CL,Round 1,Real Madrid,Esp,Napoli,Ita,2-0,1-1,3-1,no,no,Golden Era,stage
4,1987/88,CL,Round 1,Girondins Bordeaux,Fra,BFC Dynamo Berlin,GDR,2-0,2-0,4-0,no,no,Golden Era,stage
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2186,1999/00,EL,Quarter Finals,Celta de Vigo,Esp,RC Lens,Fra,0-0,1-2,1-2,no,no,Golden Era,Quarter Finals
2187,1999/00,EL,Quarter Finals,Leeds United,Eng,Slavia Praha,Cze,3-0,1-2,4-2,no,no,Golden Era,Quarter Finals
2188,1999/00,EL,Semi Finals,Galatasaray,Tur,Leeds United,Eng,2-0,2-2,4-2,no,no,Golden Era,Semi Finals
2189,1999/00,EL,Semi Finals,Arsenal,Eng,RC Lens,Fra,1-0,2-1,3-1,no,no,Golden Era,Semi Finals


### 2. Per-country match-level aggregates

Collapses the Golden Era `country_stats` (one row per country per season) into one summary row per country.

Metrics produced

`avg_win_rate`: mean win rate across Golden Era seasons
`avg_ppg3_pt` : mean PPG (3pt system) per team per season
`avg_gdpg_pt` : mean goal-difference per game per team per season
`n_seasons`: how many Golden Era seasons the country participated in
`avg_num_teams`: mean number of teams entered per season

The per-team normalisation (on `num_teams`) is applied season-by-season before averaging, so a country that entered 6 teams in one season and 2 in another is not unfairly rewarded.

In [7]:
import numpy as np
import pandas as pd


_REQUIRED = ["country", "season", "num_teams",
             "win_rate", "ppg_3", "ppg_2", "gdpg"]


def country_match_aggregates(ge_cs: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate Golden Era country_stats to one row per country.

    Parameters
    ----------
    ge_cs : Golden Era country_stats  (output of step2a filter_golden_era['cs'])

    Returns
    -------
    agg : DataFrame, one row per country, sorted by avg_win_rate descending
    """
    missing = [c for c in _REQUIRED if c not in ge_cs.columns]
    if missing:
        raise KeyError(f"Missing columns in ge_cs: {missing}")

    df = ge_cs.copy()

    # Per-team metrics computed season-by-season before aggregating
    teams = df["num_teams"].replace(0, np.nan)
    df["ppg3_pt"] = df["ppg_3"] / teams
    df["ppg2_pt"] = df["ppg_2"] / teams
    df["gdpg_pt"] = df["gdpg"]  / teams

    agg = (df.groupby("country", sort=False)
             .agg(
                 n_seasons      = ("season",    "nunique"),
                 avg_num_teams  = ("num_teams", "mean"),
                 avg_win_rate   = ("win_rate",  "mean"),
                 avg_ppg3_pt    = ("ppg3_pt",   "mean"),
                 avg_ppg2_pt    = ("ppg2_pt",   "mean"),
                 avg_gdpg_pt    = ("gdpg_pt",   "mean"),
                 # Totals (useful context, not used in composite)
                 total_wins     = ("wins",          "sum"),
                 total_draws    = ("draws",         "sum"),
                 total_losses   = ("losses",        "sum"),
                 total_matches  = ("total_matches", "sum"),
             )
             .reset_index()
             .sort_values("avg_win_rate", ascending=False)
             .reset_index(drop=True))

    # NaN-safe rank: countries with no win_rate data get NaN rank, not a crash
    agg["rank_win_rate"] = (agg["avg_win_rate"]
                            .rank(ascending=False, method="min", na_option="bottom")
                            .astype("Int64"))   # nullable integer — holds NaN safely

    # ── Warn if any countries have incomplete data ────────────────────────────
    null_rows = agg[agg["avg_win_rate"].isna()]
    if not null_rows.empty:
        print(f"\n  WARNING: {len(null_rows)} country/countries have NaN avg_win_rate "
              f"(no match data in Golden Era) — ranked last:")
        print("  " + ", ".join(null_rows["country"].tolist()))

    # ── Print top-10 preview ──────────────────────────────────────────────────
    def _fmt(val, fmt=".3f"):
        """Format a value that might be NaN."""
        return f"{val:{fmt}}" if pd.notna(val) else "  —"

    print("\n  Per-country match aggregates (Golden Era) — top 10 by win rate")
    print(f"  {'#':>3}  {'Country':<20}  {'Seasons':>7}  "
          f"{'Win rate':>9}  {'PPG/team':>9}  {'GDpG/team':>10}")
    print("  " + "-" * 65)
    for _, r in agg.head(10).iterrows():
        print(f"  {r['rank_win_rate']:>3}  {r['country']:<20}  "
              f"{int(r['n_seasons']):>7}  "
              f"{_fmt(r['avg_win_rate']):>9}  "
              f"{_fmt(r['avg_ppg3_pt']):>9}  "
              f"{_fmt(r['avg_gdpg_pt']):>10}")

    # ── Highlight Italy ───────────────────────────────────────────────────────
    italy = agg[agg["country"].str.upper() == "ITA"]
    if not italy.empty:
        r = italy.iloc[0]
        print(f"\n  Italy  →  rank #{r['rank_win_rate']} of "
              f"{len(agg)} countries  "
              f"(win rate {_fmt(r['avg_win_rate'])})")

    return agg

In [8]:
ge["cs"].head(3)

,country,season,num_teams,wins,draws,losses,goals_for,goals_against,total_matches,win_rate,draw_rate,loss_rate,ppg_3,ppg_2,goal_diff,gf_pg,ga_pg,gdpg,era
0,Alb,1987/88,3,5,0,6,13.0,13.0,11,0.454545,0.0,0.545455,1.363636,0.909091,0.0,1.181818,1.181818,0.000000,Golden Era
1,Alb,1988/89,2,1,0,5,5.0,10.0,6,0.166667,0.0,0.833333,0.500000,0.333333,-5.0,0.833333,1.666667,-0.833333,Golden Era
2,Alb,1989/90,3,3,0,7,12.0,20.0,10,0.300000,0.0,0.700000,0.900000,0.600000,-8.0,1.200000,2.000000,-0.800000,Golden Era


In [9]:
match_agg = country_match_aggregates(ge["cs"])


  Gib, Kaz, Kos, Mon, Sma

  Per-country match aggregates (Golden Era) — top 10 by win rate
    #  Country               Seasons   Win rate   PPG/team   GDpG/team
  -----------------------------------------------------------------
    1  Ita                        13      0.545      0.275       0.114
    2  Esp                        13      0.517      0.299       0.110
    3  Ger                        13      0.498      0.264       0.096
    4  Fra                        13      0.492      0.310       0.111
    5  YUG                        13      0.482      0.389       0.093
    6  Svk                        13      0.476      0.490      -0.019
    7  Eng                        13      0.464      0.383       0.141
    8  Geo                        13      0.439      0.896      -0.089
    9  Ukr                        13      0.439      0.455       0.003
   10  Rus                        13      0.427      0.332       0.028

  Italy  →  rank #1 of 59 countries  (win rate 0.545)


### 3. Per-country tournament depth

Uses country_season_highest (Golden Era slice) to compute:

- `avg_round_ord` : mean ordinal depth across seasons (stage=0, R16=1, QF=2, SF=3, Final=4)
- `max_round_ord` : best round ever reached
- `n_finals` : seasons the country reached the Final
- `n_semis` : seasons reaching the Semi Finals
- `n_qf` : seasons reaching the Quarter Finals
- `finals_rate` : n_finals / n_seasons  (comparable across countries with different participation lengths)

In [10]:

import numpy as np
import pandas as pd

# from step0_round import ROUND_GROUPS   # ['stage','Round of 16','Quarter Finals','Semi Finals','Final']


def _ordinal(series: pd.Series) -> pd.Series:
    """
    Convert highest_round strings to integer ordinal (0-4). Unknown/null -> NaN.

    Values not in ROUND_GROUPS are replaced with NaN *before* building the
    Categorical, which avoids the Pandas4Warning about non-null entries not
    in the dtype's categories.
    """
    # Replace any value not in ROUND_GROUPS with NaN before categorising
    ROUND_GROUPS = ["stage", "Round of 16", "Quarter Finals", "Semi Finals", "Final"]

    known = set(ROUND_GROUPS)
    clean = series.where(series.isin(known), other=np.nan)

    # Warn once if unknown labels were found
    unknown = series[series.notna() & ~series.isin(known)]
    if not unknown.empty:
        counts = unknown.value_counts()
        print(f"  [depth] WARNING - {len(unknown)} row(s) have unrecognised "
              f"highest round values -> treated as NaN:")
        for val, n in counts.items():
            print(f"    '{val}'  ({n} row(s))")

    cat   = pd.Categorical(clean, categories=ROUND_GROUPS, ordered=True)
    codes = pd.Series(cat.codes, index=series.index, dtype="float")
    return codes.replace(-1, np.nan)


def country_depth_aggregates(ge_csh: pd.DataFrame,
                              round_col: str = "highest round") -> pd.DataFrame:
    """
    Aggregate Golden Era country_season_highest to one row per country.

    Parameters
    ----------
    ge_csh    : Golden Era country_season_highest
    round_col : column holding the grouped round label

    Returns
    -------
    agg : DataFrame, one row per country, sorted by avg_round_ord descending
    """
    ROUND_GROUPS = ["stage", "Round of 16", "Quarter Finals", "Semi Finals", "Final"]

    if round_col not in ge_csh.columns:
        raise KeyError(f"Column '{round_col}' not found in ge_csh.")

    df = ge_csh.copy()
    df["round_ord"] = _ordinal(df[round_col])

    agg = (df.groupby("country", sort=False)
             .agg(
                 n_seasons     = ("season",    "nunique"),
                 avg_round_ord = ("round_ord", "mean"),
                 max_round_ord = ("round_ord", "max"),
                 n_finals      = (round_col, lambda x: (x == "Final").sum()),
                 n_semis       = (round_col, lambda x: (x == "Semi Finals").sum()),
                 n_qf          = (round_col, lambda x: (x == "Quarter Finals").sum()),
             )
             .reset_index())

    agg["finals_rate"] = agg["n_finals"] / agg["n_seasons"]
    agg["sf_rate"]     = agg["n_semis"]  / agg["n_seasons"]

    # Best round label for display — NaN-safe
    agg["best_round"] = agg["max_round_ord"].apply(
        lambda x: ROUND_GROUPS[int(x)] if pd.notna(x) else "--"
    )

    agg = (agg.sort_values("avg_round_ord", ascending=False)
              .reset_index(drop=True))

    # NaN-safe rank: nullable Int64 so countries with no depth data don't crash
    agg["rank_depth"] = (agg["avg_round_ord"]
                         .rank(ascending=False, method="min", na_option="bottom")
                         .astype("Int64"))

    # Warn if any countries have no depth data at all
    null_rows = agg[agg["avg_round_ord"].isna()]
    if not null_rows.empty:
        print(f"\n  WARNING: {len(null_rows)} country/countries have NaN "
              f"avg_round_ord (no depth data in Golden Era) -- ranked last:")
        print("  " + ", ".join(null_rows["country"].tolist()))

    def _fmt(val, fmt=".3f"):
        return f"{val:{fmt}}" if pd.notna(val) else "  --"

    # ── Print top-10 preview ──────────────────────────────────────────────────
    print("\n  Per-country tournament depth (Golden Era) -- top 10 by avg round")
    print(f"  {'#':>3}  {'Country':<20}  {'N':>4}  "
          f"{'Avg round':>10}  {'Finals':>7}  {'Semis':>6}  "
          f"{'F-rate':>7}  {'Best':>15}")
    print("  " + "-" * 78)
    for _, r in agg.head(10).iterrows():
        print(f"  {r['rank_depth']:>3}  {r['country']:<20}  "
              f"{int(r['n_seasons']):>4}  "
              f"{_fmt(r['avg_round_ord']):>10}  "
              f"{int(r['n_finals']):>7}  "
              f"{int(r['n_semis']):>6}  "
              f"{_fmt(r['finals_rate']):>7}  "
              f"{r['best_round']:>15}")

    italy = agg[agg["country"].str.upper() == "ITA"]
    if not italy.empty:
        r = italy.iloc[0]
        print(f"\n  Italy  ->  rank #{r['rank_depth']} of {len(agg)} countries  "
              f"(avg round ord {_fmt(r['avg_round_ord'])}, "
              f"{int(r['n_finals'])} finals)")

    return agg

In [11]:
ge["csh"].columns

Index(['season', 'country', 'highest round', 'competition', 'era'], dtype='str')

In [12]:
depth_agg = country_depth_aggregates(ge["csh"])

  [depth] WARNING - 233 row(s) have unrecognised highest round values -> treated as NaN:
    'not partecipate'  (233 row(s))

  Gib, Kaz, Kos, Mon, Sma

  Per-country tournament depth (Golden Era) -- top 10 by avg round
    #  Country                  N   Avg round   Finals   Semis   F-rate             Best
  ------------------------------------------------------------------------------
    1  Ita                     13       3.769       11       1    0.846            Final
    2  Esp                     13       3.692       10       2    0.769            Final
    3  Ger                     13       3.615        8       5    0.615            Final
    4  Fra                     13       3.385        6       6    0.462            Final
    5  Eng                     13       3.100        6       1    0.462            Final
    6  Ned                     13       2.231        4       1    0.308            Final
    7  Bel                     13       1.846        3       2    0.231     

In [13]:
depth_agg.head()

,country,n_seasons,avg_round_ord,max_round_ord,n_finals,n_semis,n_qf,finals_rate,sf_rate,best_round,rank_depth
0,Ita,13,3.769231,4.0,11,1,1,0.846154,0.076923,Final,1
1,Esp,13,3.692308,4.0,10,2,1,0.769231,0.153846,Final,2
2,Ger,13,3.615385,4.0,8,5,0,0.615385,0.384615,Final,3
3,Fra,13,3.384615,4.0,6,6,1,0.461538,0.461538,Final,4
4,Eng,13,3.100000,4.0,6,1,2,0.461538,0.076923,Final,5


### 4. Per-competition breakdown

Uses `country_season_competition_highest` (Golden Era slice).

For each competition (CL, CW, EL, ...) produces a per-country table of:

- `avg_round_ord` : mean ordinal depth across seasons in that competition
- `n_finals` : Finals reached in that competition
- `n_seasons` : seasons the country participated in that competition
- `rank` : country rank within that competition

Also produces a wide pivot table (country × competition) useful for
the heatmap in the visualisation step.

Competition codes expected:
- 'CL'  Champions League / European Cup
- 'CW'  Cup Winners' Cup
- 'EL'  UEFA Cup / Europa League
- 'CO'  Conference League  (if present)

In [14]:
import numpy as np
import pandas as pd


def _ordinal(series: pd.Series) -> pd.Series:
    """
    Convert highest_round strings to integer ordinal (0-4). Unknown/null -> NaN.

    Unknown values are replaced with NaN *before* building the Categorical,
    avoiding the Pandas4Warning about non-null entries not in dtype categories.
    """
    ROUND_GROUPS = ["stage", "Round of 16", "Quarter Finals", "Semi Finals", "Final"]
    known = set(ROUND_GROUPS)
    clean = series.where(series.isin(known), other=np.nan)

    unknown = series[series.notna() & ~series.isin(known)]
    if not unknown.empty:
        counts = unknown.value_counts()
        print(f"  [comp] WARNING - {len(unknown)} row(s) have unrecognised "
              f"highest_round values -> treated as NaN:")
        for val, n in counts.items():
            print(f"    '{val}'  ({n} row(s))")

    cat = pd.Categorical(clean, categories=ROUND_GROUPS, ordered=True)
    return pd.Series(cat.codes, index=series.index, dtype="float").replace(-1, np.nan)


def _fmt(val, fmt=".3f") -> str:
    """Format a value that might be NaN."""
    return f"{val:{fmt}}" if pd.notna(val) else "  --"


def country_competition_breakdown(
    ge_csch: pd.DataFrame,
    round_col:   str = "highest_round",
    comp_col:    str = "competition",
    country_col: str = "country",
) -> tuple[dict[str, pd.DataFrame], pd.DataFrame]:
    """
    Per-competition country rankings for the Golden Era.

    Parameters
    ----------
    ge_csch      : Golden Era country_season_competition_highest
    round_col    : column with grouped round label
    comp_col     : column with competition code
    country_col  : column with country name

    Returns
    -------
    by_comp : dict  {competition_code -> per-country DataFrame}
    pivot   : wide DataFrame  (country x competition, values = avg_round_ord)
              ready for heatmap plotting
    """
    ROUND_GROUPS = ["stage", "Round of 16", "Quarter Finals", "Semi Finals", "Final"]
    if round_col not in ge_csch.columns:
        raise KeyError(f"'{round_col}' not found in ge_csch.")

    df = ge_csch.copy()
    df["round_ord"] = _ordinal(df[round_col])

    competitions = sorted(df[comp_col].dropna().unique())
    by_comp      = {}

    print("\n  Per-competition breakdown (Golden Era)")
    print("  " + "=" * 72)

    for comp in competitions:
        sub = df[df[comp_col] == comp].copy()

        agg = (sub.groupby(country_col, sort=False)
                  .agg(
                      n_seasons     = ("season",    "nunique"),
                      avg_round_ord = ("round_ord", "mean"),
                      max_round_ord = ("round_ord", "max"),
                      n_finals      = (round_col, lambda x: (x == "Final").sum()),
                      n_semis       = (round_col, lambda x: (x == "Semi Finals").sum()),
                  )
                  .reset_index()
                  .sort_values("avg_round_ord", ascending=False)
                  .reset_index(drop=True))

        # NaN-safe rank
        agg["rank"] = (agg["avg_round_ord"]
                       .rank(ascending=False, method="min", na_option="bottom")
                       .astype("Int64"))

        # NaN-safe best round label
        agg["best_round"] = agg["max_round_ord"].apply(
            lambda x: ROUND_GROUPS[int(x)] if pd.notna(x) else "--"
        )

        by_comp[comp] = agg

        # ── Print per-competition top-8 ───────────────────────────────────────
        italy = agg[agg[country_col].str.upper() == "ITA"]
        if not italy.empty:
            italy_rank = f"#{italy['rank'].values[0]}"
            italy_avg  = _fmt(italy["avg_round_ord"].values[0], ".2f")
        else:
            italy_rank, italy_avg = "N/A", "--"

        print(f"\n  [{comp}]  Italy: rank {italy_rank} of {len(agg)} "
              f"| avg round ord {italy_avg}")
        print(f"  {'#':>3}  {'Country':<20}  {'N':>4}  "
              f"{'Avg round':>10}  {'Finals':>7}  {'Semis':>6}")
        print("  " + "-" * 55)
        for _, r in agg.head(8).iterrows():
            print(f"  {r['rank']:>3}  {r[country_col]:<20}  "
                  f"{int(r['n_seasons']):>4}  "
                  f"{_fmt(r['avg_round_ord']):>10}  "
                  f"{int(r['n_finals']):>7}  "
                  f"{int(r['n_semis']):>6}")

    # ── Pivot table (country x competition) ──────────────────────────────────
    pivot_rows = []
    for comp, agg in by_comp.items():
        for _, r in agg.iterrows():
            pivot_rows.append({
                country_col:    r[country_col],
                "competition":  comp,
                "avg_round_ord": r["avg_round_ord"],
            })

    pivot = (pd.DataFrame(pivot_rows)
               .pivot_table(index=country_col,
                            columns="competition",
                            values="avg_round_ord",
                            aggfunc="mean"))
    pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

    print(f"\n  Pivot table shape: {pivot.shape}  "
          f"(countries x competitions)")

    return by_comp, pivot

In [15]:
ge["csch"].head()

,season,country,competition,highest_round,era
0,1987/88,Alb,CL,stage,Golden Era
1,1987/88,Alb,CO,not partecipate,Golden Era
2,1987/88,Alb,CW,stage,Golden Era
3,1987/88,Alb,EL,stage,Golden Era
4,1987/88,Alb,INTER-CITIES FAIRS CUP,not partecipate,Golden Era


In [16]:
by_comp, pivot = country_competition_breakdown(ge["csch"])

  [comp] WARNING - 2418 row(s) have unrecognised highest_round values -> treated as NaN:
    'not partecipate'  (2418 row(s))

  Per-competition breakdown (Golden Era)

  [CL]  Italy: rank #1 of 59 | avg round ord 3.31
    #  Country                  N   Avg round   Finals   Semis
  -------------------------------------------------------
    1  Ita                     13       3.308        9       1
    2  Esp                     13       2.462        4       2
    3  Fra                     13       2.231        2       5
    3  Ger                     13       2.231        2       5
    5  Ned                     13       1.583        3       1
    6  Por                     13       1.462        2       1
    7  Eng                     13       1.222        1       1
    8  URS                     13       1.000        0       1

  [CO]  Italy: rank #1 of 59 | avg round ord   --
    #  Country                  N   Avg round   Finals   Semis
  ----------------------------------------

In [ ]:
by_comp.keys()

### 5. Composite score and final ranking

Merges the match-level aggregates and depth aggregates (step2c), applies min-max normalisation within the Golden Era, then computes a weighted composite dominance score.

Weights:
- `avg_round_ord` 35 % (tournament depth — clearest dominance signal)
- `avg_win_rate` 25 % (match efficiency)
- `avg_ppg3_pt` 25 % (points accumulation, normalised by quota)
- `avg_gdpg_pt` 15 % (quality of wins / goal margin)

These are the defaults, we can modify the weights by passing a custom weight dict to override defaults.

In [17]:
import numpy as np
import pandas as pd


DEFAULT_WEIGHTS = {
    "avg_round_ord" : 0.35,
    "avg_win_rate"  : 0.25,
    "avg_ppg3_pt"   : 0.25,
    "avg_gdpg_pt"   : 0.15,
}

_REQUIRED_MATCH = ["country", "avg_win_rate", "avg_ppg3_pt", "avg_gdpg_pt"]
_REQUIRED_DEPTH = ["country", "avg_round_ord", "n_finals", "n_semis",
                   "finals_rate", "n_seasons"]


def _minmax_norm(series: pd.Series) -> pd.Series:
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.0, index=series.index)
    return (series - mn) / (mx - mn)


def build_composite(
    match_agg: pd.DataFrame,
    depth_agg: pd.DataFrame,
    weights:   dict[str, float] | None = None,
) -> pd.DataFrame:
    """
    Merge, normalise and rank all countries by composite score.

    Parameters
    ----------
    match_agg : output of step2b country_match_aggregates()
    depth_agg : output of step2c country_depth_aggregates()
    weights   : optional override dict  {column_name: weight}
                Must sum to 1.0.

    Returns
    -------
    rankings : DataFrame, one row per country, sorted by composite_score desc
               Includes all raw metrics, normalised pillars, and final rank.
    """
    if weights is None:
        weights = DEFAULT_WEIGHTS

    total_w = sum(weights.values())
    if abs(total_w - 1.0) > 1e-6:
        raise ValueError(f"Weights must sum to 1.0, got {total_w:.4f}.")

    missing_m = [c for c in _REQUIRED_MATCH if c not in match_agg.columns]
    missing_d = [c for c in _REQUIRED_DEPTH if c not in depth_agg.columns]
    if missing_m or missing_d:
        raise KeyError(f"Missing cols — match: {missing_m}, depth: {missing_d}. "
                       "Run step2b and step2c first.")

    # ── Merge on country ──────────────────────────────────────────────────────
    merged = match_agg.merge(
        depth_agg[["country", "avg_round_ord", "max_round_ord",
                   "n_finals", "n_semis", "n_qf",
                   "finals_rate", "sf_rate", "best_round",
                   "n_seasons"]],
        on="country", how="outer", suffixes=("_match", "_depth"),
    )

    # If a country appears in one source but not the other, flag it
    match_only = merged["avg_win_rate"].isna().sum()
    depth_only = merged["avg_round_ord"].isna().sum()
    if match_only:
        print(f"  WARNING: {match_only} countries have depth data but no match data.")
    if depth_only:
        print(f"  WARNING: {depth_only} countries have match data but no depth data.")

    # ── Min-max normalise each pillar ────────────────────────────────────────
    norm_cols = {}
    for col in weights:
        if col not in merged.columns:
            raise KeyError(f"Weight column '{col}' not found after merge.")
        norm_col = f"{col}_norm"
        merged[norm_col] = _minmax_norm(merged[col].fillna(merged[col].median()))
        norm_cols[col] = norm_col

    # ── Composite score ───────────────────────────────────────────────────────
    merged["composite_score"] = sum(
        weights[col] * merged[norm_cols[col]]
        for col in weights
    )

    # ── Final ranking ────────────────────────────────────────────────────────
    merged = (merged.sort_values("composite_score", ascending=False)
                    .reset_index(drop=True))
    merged["rank"] = range(1, len(merged) + 1)

    # ── Print summary ─────────────────────────────────────────────────────────
    print("\n  Final country rankings (Golden Era) — top 15 by composite score")
    print(f"  Weights: " +
          "  ".join(f"{k.replace('avg_','').replace('_ord','').replace('_pt','')}"
                    f"={v:.0%}" for k, v in weights.items()))
    print(f"\n  {'Rk':>3}  {'Country':<20}  {'Score':>7}  "
          f"{'WinRate':>8}  {'PPG/tm':>7}  {'GDpG/tm':>8}  "
          f"{'AvgRnd':>7}  {'Finals':>7}")
    print("  " + "-" * 75)

    for _, r in merged.head(15).iterrows():
        print(f"  {int(r['rank']):>3}  {r['country']:<20}  "
              f"{r['composite_score']:>7.4f}  "
              f"{r['avg_win_rate']:>8.3f}  "
              f"{r['avg_ppg3_pt']:>7.3f}  "
              f"{r['avg_gdpg_pt']:>8.3f}  "
              f"{r['avg_round_ord']:>7.3f}  "
              f"{int(r['n_finals'] if not pd.isna(r['n_finals']) else 0):>7}")

    italy = merged[merged["country"].str.upper() == "ITA"]
    if not italy.empty:
        r = italy.iloc[0]
        print(f"\n  ► Italy  →  rank #{int(r['rank'])} of {len(merged)}  "
              f"| composite score {r['composite_score']:.4f}")
        print(f"    win rate {r['avg_win_rate']:.3f}  "
              f"ppg/team {r['avg_ppg3_pt']:.3f}  "
              f"gdpg/team {r['avg_gdpg_pt']:.3f}  "
              f"avg round {r['avg_round_ord']:.3f}  "
              f"finals {int(r['n_finals'])}")

    return merged

In [18]:
rankings = build_composite(match_agg, depth_agg)


  Final country rankings (Golden Era) — top 15 by composite score
  Weights: round=35%  win_rate=25%  ppg3=25%  gdpg=15%

   Rk  Country                 Score   WinRate   PPG/tm   GDpG/tm   AvgRnd   Finals
  ---------------------------------------------------------------------------
    1  Ita                    0.8262     0.545    0.275     0.114    3.769       11
    2  Esp                    0.8128     0.517    0.299     0.110    3.692       10
    3  Ger                    0.7869     0.498    0.264     0.096    3.615        8
    4  Fra                    0.7757     0.492    0.310     0.111    3.385        6
    5  Eng                    0.7575     0.464    0.383     0.141    3.100        6
    6  Ned                    0.6279     0.410    0.302     0.064    2.231        4
    7  Bel                    0.6007     0.424    0.309     0.050    1.846        3
    8  Geo                    0.5972     0.439    0.896    -0.089    0.000        0
    9  YUG                    0.5898     0.

### 6. Statistical tests - Italy vs Europe

Two complementary tests for each metric:
1. Mann-Whitney $U$  (one-tailed)
    - Italy's season-by-season values vs the pooled distribution of all other countries in the same Golden Era seasons. $H_{1}$: Italy > Europe
2. Season rank consistency
    - Each season, rank all countries by the metric. Report Italy's mean rank and compare against expectation (expectation = median rank if all countries were equal).

Both tests use the raw season-level data from the Golden Era country_stats slice, NOT the already-aggregated tables from 2. /3. . This preserves the full within-era variance.

In [19]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

ROUND_GROUPS = ["stage", "Round of 16", "Quarter Finals", "Semi Finals", "Final"]


# ── Shared helpers ────────────────────────────────────────────────────────────

def _cohen_d(a: np.ndarray, b: np.ndarray) -> float:
    var_a = np.var(a, ddof=1) if len(a) > 1 else 0.0
    var_b = np.var(b, ddof=1) if len(b) > 1 else 0.0
    pooled = np.sqrt((var_a + var_b) / 2)
    return (np.mean(a) - np.mean(b)) / pooled if pooled > 0 else 0.0

def _sig_stars(p: float) -> str:
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.10: return "*"
    return "ns"

def _effect_label(d: float) -> str:
    a = abs(d)
    if a < 0.2: return "negligible"
    if a < 0.5: return "small"
    if a < 0.8: return "medium"
    return "large"


MIN_N = 3


# Test 1: Mann-Whitney U — Italy vs pooled rest

def test_italy_vs_rest_match(
    ge_cs: pd.DataFrame,
    country_col: str = "country",
    country_name: str = "Ita",
) -> pd.DataFrame:
    """
    For each match-level metric, test whether Italy's season-by-season
    values are significantly higher than those of all other countries.

    Parameters
    ----------
    ge_cs        : Golden Era country_stats with derived per-team metrics
                   (must have ppg3_pt and gdpg_pt — run step2b logic first,
                    or add_derived_metrics if using the full cs slice)
    country_col  : country column name
    country_name : country to isolate

    Returns
    -------
    results DataFrame with one row per metric
    """
    # Add per-team cols if not already present
    df = ge_cs.copy()
    if "ppg3_pt" not in df.columns:
        teams = df["num_teams"].replace(0, np.nan)
        df["ppg3_pt"] = df["ppg_3"] / teams
        df["gdpg_pt"] = df["gdpg"]  / teams

    name_up  = country_name.strip().upper()
    italy    = df[df[country_col].str.upper() == name_up]
    rest     = df[df[country_col].str.upper() != name_up]

    metrics = {
        "win_rate" : "Win rate",
        "ppg3_pt"  : "PPG (3pt) per team",
        "gdpg_pt"  : "GDpG per team",
    }

    rows = []
    print(f"\n --- Mann-Whitney U: {country_name} vs rest --- ")
    print(f"  {'Metric':<22}  {'n_italy':>7}  {'n_rest':>7}  "
          f"{'U':>8}  {'p':>8}  {'sig':>4}  {'d':>6}  effect")
    print("  " + "-" * 80)

    for col, label in metrics.items():
        italy_vals = italy[col].dropna().values
        rest_vals  = rest [col].dropna().values
        row = {"metric": label, "n_italy": len(italy_vals),
               "n_rest": len(rest_vals)}

        if len(italy_vals) < MIN_N or len(rest_vals) < MIN_N:
            row.update(U=np.nan, p=np.nan, sig="skip", d=np.nan, effect="too few")
            rows.append(row)
            continue

        stat, p = mannwhitneyu(italy_vals, rest_vals, alternative="greater")
        d       = _cohen_d(italy_vals, rest_vals)
        row.update(U=stat, p=p, sig=_sig_stars(p), d=d, effect=_effect_label(d))
        rows.append(row)
        print(f"  {label:<22}  {len(italy_vals):>7}  {len(rest_vals):>7}  "
              f"{stat:>8.0f}  {p:>8.4f}  {_sig_stars(p):>4}  "
              f"{d:>+6.2f}  {_effect_label(d)}")

    print()
    return pd.DataFrame(rows)


def test_italy_vs_rest_depth(
    ge_csh: pd.DataFrame,
    # round_col:    str = "highest_round",
    round_col:    str = "highest round",
    country_col:  str = "country",
    country_name: str = "Ita",
) -> pd.DataFrame:
    """
    Test whether Italy's season-by-season round depth is significantly
    higher than all other countries' depths.

    Parameters
    ----------
    ge_csh       : Golden Era country_season_highest

    Returns
    -------
    results DataFrame with one row
    """
    df = ge_csh.copy()
    cat = pd.Categorical(df[round_col], categories=ROUND_GROUPS, ordered=True)
    df["round_ord"] = pd.Series(cat.codes, dtype="float").replace(-1, np.nan)

    name_up    = country_name.strip().upper()
    italy_vals = df[df[country_col].str.upper() == name_up]["round_ord"].dropna().values
    rest_vals  = df[df[country_col].str.upper() != name_up]["round_ord"].dropna().values

    row = {"metric": "Round depth (ord)",
           "n_italy": len(italy_vals), "n_rest": len(rest_vals)}

    print(f"  --- Mann-Whitney U: round depth --- ")
    print(f"  {'Metric':<22}  {'n_italy':>7}  {'n_rest':>7}  "
          f"{'U':>8}  {'p':>8}  {'sig':>4}  {'d':>6}  effect")
    print("  " + "-" * 80)

    if len(italy_vals) < MIN_N or len(rest_vals) < MIN_N:
        row.update(U=np.nan, p=np.nan, sig="skip", d=np.nan, effect="too few")
        print("  SKIP — too few seasons")
    else:
        stat, p = mannwhitneyu(italy_vals, rest_vals, alternative="greater")
        d       = _cohen_d(italy_vals, rest_vals)
        row.update(U=stat, p=p, sig=_sig_stars(p), d=d, effect=_effect_label(d))
        print(f"  {'Round depth (ord)':<22}  {len(italy_vals):>7}  {len(rest_vals):>7}  "
              f"{stat:>8.0f}  {p:>8.4f}  {_sig_stars(p):>4}  "
              f"{d:>+6.2f}  {_effect_label(d)}")

    print()
    return pd.DataFrame([row])


# Test 2: Season-rank consistency 

def rank_consistency(
    ge_cs: pd.DataFrame,
    country_col:  str = "country",
    country_name: str = "Ita",
) -> pd.DataFrame:
    """
    Each Golden Era season: rank all countries by win_rate.
    Report Italy's mean rank, median rank, and how often it
    was in the top 3 / top 5.

    Returns
    -------
    season_ranks : DataFrame with one row per season showing Italy's rank
    """
    df = ge_cs.copy()
    seasons = sorted(df["season"].unique(),
                     key=lambda s: int(s.split("/")[0]))
    name_up = country_name.strip().upper()

    records = []
    for s in seasons:
        sub = df[df["season"] == s].copy()
        sub["season_rank"] = sub["win_rate"].rank(ascending=False, method="min")
        n_countries = len(sub)
        italy_row = sub[sub[country_col].str.upper() == name_up]
        if italy_row.empty:
            continue
        records.append({
            "season"     : s,
            "rank"       : int(italy_row["season_rank"].values[0]),
            "n_countries": n_countries,
            "win_rate"   : italy_row["win_rate"].values[0],
            "pct_rank"   : italy_row["season_rank"].values[0] / n_countries,
        })

    season_ranks = pd.DataFrame(records)
    if season_ranks.empty:
        print(f"  No seasons found for {country_name} — check country name.")
        return season_ranks

    mean_rank    = season_ranks["rank"].mean()
    median_rank  = season_ranks["rank"].median()
    top3_count   = (season_ranks["rank"] <= 3).sum()
    top5_count   = (season_ranks["rank"] <= 5).sum()
    n_seasons    = len(season_ranks)
    n_countries  = season_ranks["n_countries"].median()

    print(f" --- Season rank consistency: {country_name} --- ")
    print(f"  Seasons analysed : {n_seasons}")
    print(f"  Avg countries/season : {n_countries:.0f}")
    print(f"  Mean rank   : {mean_rank:.2f}  "
          f"(expected if random: {(n_countries+1)/2:.1f})")
    print(f"  Median rank : {median_rank:.0f}")
    print(f"  Top-3 finishes : {top3_count} / {n_seasons}  "
          f"({100*top3_count/n_seasons:.0f}%)")
    print(f"  Top-5 finishes : {top5_count} / {n_seasons}  "
          f"({100*top5_count/n_seasons:.0f}%)")
    print(f"\n  {'Season':<10}  {'Rank':>5}  {'/ N':>5}  {'Win rate':>9}")
    print("  " + "-" * 38)
    for _, r in season_ranks.iterrows():
        flag = " ◄ top 3" if r["rank"] <= 3 else ""
        print(f"  {r['season']:<10}  {int(r['rank']):>5}  "
              f"{int(r['n_countries']):>5}  {r['win_rate']:>9.3f}{flag}")
    print()

    return season_ranks

In [20]:
res_match = test_italy_vs_rest_match(ge["cs"])
res_depth = test_italy_vs_rest_depth(ge["csh"])
s_ranks   = rank_consistency(ge["cs"])


 --- Mann-Whitney U: Ita vs rest --- 
  Metric                  n_italy   n_rest         U         p   sig       d  effect
  --------------------------------------------------------------------------------
  Win rate                     13      521      6132    0.0000   ***   +1.77  large
  PPG (3pt) per team           13      521      3022    0.7468    ns   -0.20  small
  GDpG per team                13      521      5710    0.0000   ***   +0.70  medium

  --- Mann-Whitney U: round depth --- 
  Metric                  n_italy   n_rest         U         p   sig       d  effect
  --------------------------------------------------------------------------------
  Round depth (ord)            13      521      6366    0.0000   ***   +2.92  large

 --- Season rank consistency: Ita --- 
  Seasons analysed : 13
  Avg countries/season : 59
  Mean rank   : 4.92  (expected if random: 30.0)
  Median rank : 4
  Top-3 finishes : 4 / 13  (31%)
  Top-5 finishes : 10 / 13  (77%)

  Season       Rank  

C:\Users\olgag\AppData\Local\Temp\ipykernel_8040\3090475431.py:122: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  cat = pd.Categorical(df[round_col], categories=ROUND_GROUPS, ordered=True)


In [ ]:
s_ranks